In [1]:
import torch
import optuna, json
import numpy as np
from fem_utilities.fem_fenics import *
from scipy.sparse.linalg import spsolve
from neural_net.training import *
from neural_net.networks import FullyConnected
from sklearn import preprocessing
import pickle

In [2]:
torch.manual_seed(25)

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Extração da matriz de rigidez K e vetor fonte F

In [4]:
# pontos_laser = [(0.05, 0.05)]
# A_pontos     = 1.3e6

# pontos_laser = [(0.045, 0.045), (0.055, 0.055)]
# A_pontos     = 1.3e6/2

pontos_laser = [(0.045, 0.045), (0.055, 0.045), (0.045, 0.055), (0.055, 0.055)]
A_pontos     = 1.3e6/4

In [5]:
bioheat1 = BioHeatSimulation('mesh/malha.xdmf')

K, F = bioheat1.extract_system_matrices(pontos_laser, A_pontos)
F    = F.reshape(-1, 1)

print("Shape da matriz A:", K.shape)
print("Shape do vetor b:", F.shape)

K_torch = torch.sparse_csr_tensor(crow_indices=K.indptr, col_indices=K.indices, values=K.data, size=K.shape).to(device)
F_torch = torch.tensor(F, dtype=torch.double).to(device)

Shape da matriz A: (89, 89)
Shape do vetor b: (89, 1)


/tmp/ipykernel_17710/56870158.py:9: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  K_torch = torch.sparse_csr_tensor(crow_indices=K.indptr, col_indices=K.indices, values=K.data, size=K.shape).to(device)
/tmp/ipykernel_17710/56870158.py:9: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  K_torch = torch.sparse_csr_tensor(crow_indices=K.indptr, col_indices=K.indices, values=K.data, size=K.shape).to(device)


In [6]:
# def scipy_to_torch_sparse(A):
#     A_coo   = A.tocoo()
#     values  = A_coo.data
#     indices = np.vstack((A_coo.row, A_coo.col))
    
#     i     = torch.LongTensor(indices)
#     v     = torch.DoubleTensor(values) # Usando Double já que você usou .double() na rede
#     shape = A_coo.shape
    
#     return torch.sparse_coo_tensor(i, v, torch.Size(shape)).to(device)

# K_torch = scipy_to_torch_sparse(K)
# F_torch = torch.tensor(F, dtype=torch.double).to(device)

### Preparando dados para treinamento

In [7]:
data_in = bioheat1.extract_nodes()

scaler         = preprocessing.MinMaxScaler()
data_in_scaled = scaler.fit_transform(data_in)
pickle.dump(scaler, open('scaler.pkl', 'wb'))

data_in_torch = torch.as_tensor(data_in_scaled, dtype=torch.double).to(device)
data_in_torch.shape

torch.Size([89, 2])

### Buscando os melhores hiperparâmetros para a rede com `Optuna`

In [8]:
def objective(trial):
    # Suggest values of the hyperparameters using a trial object.
    depth    = trial.suggest_int('depth', 3, 8)
    n_hidden = trial.suggest_int('n_hidden', 32, 256)
    init     = trial.suggest_float('init', 0.1, 2)
    lr       = trial.suggest_float('lr', 1.0E-5, 1.0E-2, log=True)

    # Creation of network
    net_model        = FullyConnected(n_hidden=n_hidden, depth=depth, init=init).double().to(device)
    fem_loss_trainer = TrainFemLoss(net_model, K_torch, F_torch)
    train_loss       = fem_loss_trainer.train(data_in_torch, lr, epochs=100)

    return train_loss

# Create a study object and optimize the objective function to find the best hyperparameters.
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=500)

[I 2026-05-08 15:06:51,255] A new study created in memory with name: no-name-b0bfcea8-b9d1-4591-b947-7e30f29296f5


Epoch: 0, Train Loss: 1.61752E+02


/home/yago/anaconda3/envs/fem_nn/lib/python3.14/site-packages/optuna/study/_tell.py:65: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  float(v)
[I 2026-05-08 15:06:52,572] Trial 0 finished with value: 9.177648547993982 and parameters: {'depth': 5, 'n_hidden': 214, 'init': 0.6792848424150919, 'lr': 0.0005087113458576163}. Best is trial 0 with value: 9.177648547993982.
[I 2026-05-08 15:06:52,768] Trial 1 finished with value: 7.16363723489961 and parameters: {'depth': 5, 'n_hidden': 138, 'init': 0.20724953288351186, 'lr': 0.004152772304780553}. Best is trial 1 with value: 7.16363723489961.


Epoch: 0, Train Loss: 1.61785E+02
Epoch: 0, Train Loss: 1.61727E+02


[I 2026-05-08 15:06:53,002] Trial 2 finished with value: 7.603374286920116 and parameters: {'depth': 3, 'n_hidden': 252, 'init': 1.8302571229809326, 'lr': 0.0011512931761815072}. Best is trial 1 with value: 7.16363723489961.


...

[I 2026-05-08 15:08:58,939] Trial 497 finished with value: 9.204625244623655 and parameters: {'depth': 6, 'n_hidden': 247, 'init': 1.3689032503126077, 'lr': 0.009049704344092281}. Best is trial 421 with value: 6.445929360654859.


Epoch: 0, Train Loss: 1.61723E+02


[I 2026-05-08 15:08:59,228] Trial 498 finished with value: 7.6373296515484705 and parameters: {'depth': 6, 'n_hidden': 241, 'init': 1.4147649768731503, 'lr': 0.009731055245642143}. Best is trial 421 with value: 6.445929360654859.


Epoch: 0, Train Loss: 1.61786E+02


[I 2026-05-08 15:08:59,475] Trial 499 finished with value: 8.48452306496411 and parameters: {'depth': 6, 'n_hidden': 192, 'init': 1.5058242038230305, 'lr': 0.007703632184185988}. Best is trial 421 with value: 6.445929360654859.


In [9]:
print(study.best_params['depth'], study.best_params['n_hidden'], study.best_params['init'], study.best_params['lr'])

7 235 1.3148209864689446 0.0075029032749307075


In [10]:
# Após o study.optimize
melhores_params = study.best_params

# Adicionar informações extras úteis
dados_modelo = {
    'params': melhores_params,
    'best_value': study.best_value, # A menor perda encontrada
}

# Salvar em arquivo
with open('hyper_params.json', 'w') as f:
    json.dump(dados_modelo, f, indent=4)

print('Parâmetros salvos com sucesso!')

Parâmetros salvos com sucesso!


### Treinamento com os Hiperparâmetros determinados

In [11]:
with open('hyper_params.json', 'r') as f:
    config = json.load(f)

params = config['params']

In [12]:
model = FullyConnected(params['n_hidden'], params['depth'], params['init']).double().to(device)

In [13]:
# Treino usando FEM-NN
epoch            = 9000
fem_loss_trainer = TrainFemLoss(model, K_torch, F_torch)
train_loss       = fem_loss_trainer.train(data_in_torch, params['lr'], epochs=epoch)

Epoch: 0, Train Loss: 1.61757E+02
Epoch: 1000, Train Loss: 2.36064E+00
Epoch: 2000, Train Loss: 1.47327E+00
Epoch: 3000, Train Loss: 1.68028E-01
Epoch: 4000, Train Loss: 2.30564E-01
Epoch: 5000, Train Loss: 4.63267E-02
Epoch: 6000, Train Loss: 9.66544E-03
Epoch: 7000, Train Loss: 2.33043E-03
Epoch: 8000, Train Loss: 3.71673E-04


In [14]:
# Salvando o modelo
torch.save(model.state_dict(), 'models/model.pt')

### Utilizando o modelo salvo para fazer predições

In [15]:
# Preparando os dados para teste do modelo

bioheat2 = BioHeatSimulation('mesh/malha_fina.xdmf')

data_in_test = bioheat2.extract_nodes()

scalar              = pickle.load(open('scaler.pkl', 'rb'))
data_in_test_scaled = scalar.transform(data_in_test)
input_data          = torch.as_tensor(data_in_test_scaled, dtype=torch.double).to(device)

input_data.shape

torch.Size([1093, 2])

In [16]:
# Definindo a arquitetura do modelo e avaliando na nova malha

modelo = FullyConnected(params['n_hidden'], params['depth'], params['init']).double().to(device)
modelo.load_state_dict(torch.load('models/model.pt'))

modelo.eval()
with torch.inference_mode():
    T_femnn = modelo(input_data).detach().cpu().numpy()

T_femnn.shape

(1093, 1)

In [17]:
# Visualizando a solução com Pyvista
bioheat2.plot_solution(T_femnn)

Widget(value='<iframe src="http://localhost:44537/index.html?ui=P_0x7119e846d940_0&reconnect=auto" class="pyvi…

### Solução FEM com FEniCS

In [18]:
T_fen = bioheat2.run_simulation(pontos_laser, A_pontos)
bioheat2.plot_solution(T_fen)


Simulação Concluída
Temperatura Máxima: 46.84 °C


Widget(value='<iframe src="http://localhost:44537/index.html?ui=P_0x7119e85f3110_1&reconnect=auto" class="pyvi…

(1093, 1)